In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

In [3]:
USERNAME = 'postgres'
PASSWORD = '180404'
HOST = 'localhost'
PORT = '5433'
DB_NAME = 'subscription_performance_db'

engine = create_engine(f'postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}')

In [4]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';"))
    print(result.fetchall())

[('rev',), ('subscription',), ('account',), ('feature_usage',), ('support_ticket',), ('churn_event',), ('cal',), ('revenue_plan_tier',), ('revenue_industry',)]


## Import All Files

#### A. Subscription

In [5]:
subscription = pd.read_csv('ravenstack_subscriptions.csv')
subscription.head() 
#if end_date is null, it means the subscription is still active, then the churn_flag will automatically be true
#an account that has upgraded will have multiple records, where other than the current tier,  they will have records with 0 mrr and arr amount

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


#### B. Account

In [6]:
account = pd.read_csv('ravenstack_accounts.csv')
account.head()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


#### C. Feature Usage

In [7]:
feature_usage = pd.read_csv('ravenstack_feature_usage.csv')
feature_usage.head()

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,False
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,False


#### D. Support Tickets

In [8]:
support_ticket = pd.read_csv('ravenstack_support_tickets.csv')
support_ticket.head()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,False
3,T-dfce9a,A-4c56c9,2024-09-08,2024-09-09 23:00:00,47.0,medium,126,5.0,False
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,NaN,False


#### E. Churn Events

In [9]:
churn_event = pd.read_csv('ravenstack_churn_events.csv')
churn_event.head()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive


## Data Types Check

#### A. Subscription

In [10]:
subscription.dtypes #start_date and end_date are object, need to convert to datetime

subscription_id      object
account_id           object
start_date           object
end_date             object
plan_tier            object
seats                 int64
mrr_amount            int64
arr_amount            int64
is_trial               bool
upgrade_flag           bool
downgrade_flag         bool
churn_flag             bool
billing_frequency    object
auto_renew_flag        bool
dtype: object

In [11]:
subscription['start_date'] = pd.to_datetime(subscription['start_date'])
subscription['end_date'] = pd.to_datetime(subscription['end_date'])

In [12]:
subscription.dtypes

subscription_id              object
account_id                   object
start_date           datetime64[ns]
end_date             datetime64[ns]
plan_tier                    object
seats                         int64
mrr_amount                    int64
arr_amount                    int64
is_trial                       bool
upgrade_flag                   bool
downgrade_flag                 bool
churn_flag                     bool
billing_frequency            object
auto_renew_flag                bool
dtype: object

#### B. Account

In [13]:
account.dtypes

account_id         object
account_name       object
industry           object
country            object
signup_date        object
referral_source    object
plan_tier          object
seats               int64
is_trial             bool
churn_flag           bool
dtype: object

In [14]:
account['signup_date'] = pd.to_datetime(account['signup_date'])

In [15]:
account.dtypes

account_id                 object
account_name               object
industry                   object
country                    object
signup_date        datetime64[ns]
referral_source            object
plan_tier                  object
seats                       int64
is_trial                     bool
churn_flag                   bool
dtype: object

#### C. Feature Usage

In [16]:
feature_usage.dtypes

usage_id               object
subscription_id        object
usage_date             object
feature_name           object
usage_count             int64
usage_duration_secs     int64
error_count             int64
is_beta_feature          bool
dtype: object

In [17]:
feature_usage['usage_date'] = pd.to_datetime(feature_usage['usage_date'])

In [18]:
feature_usage.dtypes

usage_id                       object
subscription_id                object
usage_date             datetime64[ns]
feature_name                   object
usage_count                     int64
usage_duration_secs             int64
error_count                     int64
is_beta_feature                  bool
dtype: object

#### D. Support Ticket

In [19]:
support_ticket.dtypes

ticket_id                       object
account_id                      object
submitted_at                    object
closed_at                       object
resolution_time_hours          float64
priority                        object
first_response_time_minutes      int64
satisfaction_score             float64
escalation_flag                   bool
dtype: object

In [20]:
support_ticket['submitted_at'] = pd.to_datetime(support_ticket['submitted_at'])
support_ticket['closed_at'] = pd.to_datetime(support_ticket['closed_at'])

In [21]:
support_ticket.dtypes

ticket_id                              object
account_id                             object
submitted_at                   datetime64[ns]
closed_at                      datetime64[ns]
resolution_time_hours                 float64
priority                               object
first_response_time_minutes             int64
satisfaction_score                    float64
escalation_flag                          bool
dtype: object

#### E. Churn Events

In [22]:
churn_event.dtypes

churn_event_id               object
account_id                   object
churn_date                   object
reason_code                  object
refund_amount_usd           float64
preceding_upgrade_flag         bool
preceding_downgrade_flag       bool
is_reactivation                bool
feedback_text                object
dtype: object

In [23]:
churn_event['churn_date'] = pd.to_datetime(churn_event['churn_date'])
churn_event.dtypes

churn_event_id                      object
account_id                          object
churn_date                  datetime64[ns]
reason_code                         object
refund_amount_usd                  float64
preceding_upgrade_flag                bool
preceding_downgrade_flag              bool
is_reactivation                       bool
feedback_text                       object
dtype: object

## Empty String, Whitespace and NULL Check

In [24]:
# NULL Check
dataframes = {
    'subscription': subscription,
    'account': account,
    'churn_event': churn_event,
    'usage_event': feature_usage,
    'support_ticket': support_ticket
}

null_summary = []
for table, df in dataframes.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    for col, count in nulls.items():
        null_percentage = round(count / len(df) * 100, 2)
        null_summary.append({
            'table': table,
            'column': col,
            'null_count': count,
            'null_percentage': null_percentage
        })

pd.DataFrame(null_summary).sort_values(['table', 'null_count'], ascending=[True, False])

,table,column,null_count,null_percentage
1,churn_event,feedback_text,148,24.67
0,subscription,end_date,4514,90.28
2,support_ticket,satisfaction_score,825,41.25


- NULLs at feedback text means that customers skipped giving a feedback, it's an optional field, no action needed
- NULLs at end date is informative, it signals active subscription, no action needed (filling it would be misleading)
- NULLs at satisfaction score means no rating was given, no action needed (filling it with 0 would distort stats)

In [25]:
# Empty String Check
dataframes = {
    'subscription': subscription,
    'account': account,
    'churn_event': churn_event,
    'usage_event': feature_usage,
    'support_ticket': support_ticket
}

empty_summary = []
for table, df in dataframes.items():
    empties = (df == "").sum()
    empties = empties[empties > 0]
    for col, count in empties.items():
        empty_percentage = round(count / len(df) * 100, 2)
        empty_summary.append({
            'table': table,
            'column': col,
            'empty_count': count,
            'empty_percentage': empty_percentage
        })

if empty_summary:
    pd.DataFrame(empty_summary).sort_values(['table', 'empty_count'], ascending=[True, False])
else:
    print("No empty strings found.")

No empty strings found.


In [26]:
# White Space Check
dataframes = {
    'subscription': subscription,
    'account': account,
    'churn_event': churn_event,
    'usage_event': feature_usage,
    'support_ticket': support_ticket
}

whitespace_summary = []
for table, df in dataframes.items():
    whitespaces = (df.apply(lambda col: col.astype(str).str.strip() == "") & ~df.isnull() & (df != "")).sum()
    whitespaces = whitespaces[whitespaces > 0]
    for col, count in whitespaces.items():
        whitespace_percentage = round(count / len(df) * 100, 2)
        whitespace_summary.append({
            'table': table,
            'column': col,
            'whitespace_count': count,
            'whitespace_percentage': whitespace_percentage
        })
if whitespace_summary:
    pd.DataFrame(whitespace_summary).sort_values(['table', 'whitespace_count'], ascending=[True, False])
else:
    print("No whitespace-only values found.")

No whitespace-only values found.


## Duplicate Data Check

In [27]:
# 1. subscription
print(f"Subscription - duplicates: {subscription.drop(columns=['subscription_id']).duplicated().sum()}")

# 2. account
print(f"Account - duplicates: {account.drop(columns=['account_id']).duplicated().sum()}")

# 3. feature_usage
print(f"Feature Usage - duplicates: {feature_usage.drop(columns=['usage_id']).duplicated().sum()}")

# 4. support_ticket
print(f"Support Ticket - duplicates: {support_ticket.drop(columns=['ticket_id']).duplicated().sum()}")

# 5. churn_event
print(f"Churn Event - duplicates: {churn_event.drop(columns=['churn_event_id']).duplicated().sum()}")

Subscription - duplicates: 4
Account - duplicates: 0
Feature Usage - duplicates: 0
Support Ticket - duplicates: 0
Churn Event - duplicates: 0


#### A. Subscription

In [28]:
subscription['subscription_id'].duplicated().sum()

np.int64(0)

In [29]:
subscription['account_id'].duplicated().sum()

np.int64(4500)

In [30]:
# the duplicates below are exact duplicate, except for the subscription_id, so there's action needed to be taken
subscription[subscription.drop(columns=['subscription_id']).duplicated(keep=False)].sort_values('account_id')

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
1644,S-e11cde,A-0f6450,2024-12-31,NaT,Enterprise,28,5572,66864,False,False,False,False,monthly,False
4919,S-1c54b6,A-0f6450,2024-12-31,NaT,Enterprise,28,5572,66864,False,False,False,False,monthly,False
1032,S-0283b5,A-443f6f,2024-12-10,NaT,Pro,19,931,11172,False,False,False,False,annual,True
3887,S-a2fb62,A-443f6f,2024-12-10,NaT,Pro,19,931,11172,False,False,False,False,annual,True
3368,S-dee62f,A-524364,2024-12-31,NaT,Basic,12,228,2736,False,False,False,False,monthly,True
3713,S-a33656,A-524364,2024-12-31,NaT,Basic,12,228,2736,False,False,False,False,monthly,True
1171,S-da164e,A-9174e0,2024-12-21,NaT,Pro,37,1813,21756,False,False,False,False,monthly,True
3529,S-526a2c,A-9174e0,2024-12-21,NaT,Pro,37,1813,21756,False,False,False,False,monthly,True


In [31]:
#remove the duplicates from above, since the data is exactly the same, it does not matter if we keep first or keep last

subscription = subscription.drop_duplicates(subset=subscription.columns.difference(['subscription_id']), keep='first')

In [32]:
#double check if the duplicates are removed
subscription[subscription['subscription_id'] == 'S-1c54b6']

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag


In [33]:
#every columns' number of unique values
cols = ['plan_tier', 'billing_frequency', 'auto_renew_flag']
for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(subscription[col].nunique(dropna=False))


Column: plan_tier
3

Column: billing_frequency
2

Column: auto_renew_flag
2


#### B. Account

In [34]:
account['account_id'].duplicated().sum()

np.int64(0)

#### C. Feature Usage

In [35]:
feature_usage['subscription_id'].duplicated().sum()

np.int64(20033)

In [36]:
#there's no primary key for feature_usage
feature_usage['usage_id'].duplicated().sum()
feature_usage[feature_usage['usage_id'].duplicated(keep=False)].sort_values(['usage_id', 'usage_date']).head(5)

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
19294,U-0c9318,S-01b2dc,2023-12-11,feature_9,5,1060,3,False
17533,U-0c9318,S-0ffab0,2024-01-30,feature_11,3,1614,0,False
20588,U-13ce5b,S-9b623b,2023-03-26,feature_28,10,2050,1,False
9626,U-13ce5b,S-8b0950,2024-09-10,feature_9,8,824,0,True
18480,U-2103bb,S-7fc49b,2023-04-18,feature_12,9,5022,1,False


In [37]:
#adding a usage_id column as primary key since there's no unique identifier for feature_usage
feature_usage = feature_usage.drop(columns='usage_id')
feature_usage.insert(0, 'usage_id', range(1, len(feature_usage) + 1))

In [38]:
feature_usage

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,1,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,2,S-c25263,2023-08-07,feature_5,9,369,0,False
2,3,S-f29e7f,2023-12-07,feature_3,9,1458,0,False
3,4,S-be655e,2024-07-28,feature_40,5,2085,0,False
4,5,S-f9b1d0,2024-12-02,feature_12,12,900,0,False
...,...,...,...,...,...,...,...,...
24995,24996,S-c249fb,2023-07-08,feature_16,7,4116,0,False
24996,24997,S-b83d8d,2023-03-29,feature_31,8,2240,1,False
24997,24998,S-ad7716,2024-10-03,feature_5,5,2745,0,False
24998,24999,S-dbad62,2024-06-25,feature_5,7,1715,0,False


#### D. Support Ticket

In [39]:
support_ticket['ticket_id'].duplicated().sum()

np.int64(0)

#### E. Churn Events

In [40]:
churn_event['churn_event_id'].duplicated().sum()

np.int64(0)

## Data Cardinality Check

#### A. Subscription

In [41]:
# subscription table categorical columns
cols = ['plan_tier', 'billing_frequency', 'auto_renew_flag']

for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(subscription[col].value_counts(dropna=False))


Column: plan_tier
plan_tier
Enterprise    1722
Pro           1673
Basic         1601
Name: count, dtype: int64

Column: billing_frequency
billing_frequency
monthly    2536
annual     2460
Name: count, dtype: int64

Column: auto_renew_flag
auto_renew_flag
True     4002
False     994
Name: count, dtype: int64


In [42]:
# subscription id columns
cols = ['subscription_id', 'account_id',] # there are 4996 unique subscription_id belonging to 500 unique accounts

for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(subscription[col].nunique(dropna=False))


Column: subscription_id
4996

Column: account_id
500


In [43]:
subscription[['subscription_id', 'account_id']].groupby('account_id').count().sort_values('subscription_id', ascending=False).head(10)

,subscription_id
account_id,
A-d4ac0e,19
A-5a92e7,19
A-726cfa,19
A-592832,19
A-8bde0c,18
A-5a215a,18
A-692f18,17
A-82d8a6,17
A-6f8ad2,17


In [44]:
subscription[subscription['account_id'] == 'A-0a62f5']

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
761,S-561850,A-0a62f5,2024-10-23,NaT,Pro,16,784,9408,False,False,False,False,annual,True
2836,S-d8c527,A-0a62f5,2024-12-21,NaT,Basic,25,475,5700,False,False,False,False,monthly,True
3636,S-e7a190,A-0a62f5,2024-12-06,NaT,Basic,22,418,5016,False,False,False,False,annual,True
4153,S-c62bb7,A-0a62f5,2024-09-16,NaT,Enterprise,16,3184,38208,False,False,False,False,monthly,True


In [45]:
subscription.shape

(4996, 14)

#### B. Account

In [46]:
# account categorical columns
cols = ['industry', 'country', 'referral_source', 'plan_tier']

for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(account[col].value_counts(dropna=False))


Column: industry
industry
DevTools         113
FinTech          112
Cybersecurity    100
HealthTech        96
EdTech            79
Name: count, dtype: int64

Column: country
country
US    291
UK     58
IN     49
AU     32
DE     25
CA     23
FR     22
Name: count, dtype: int64

Column: referral_source
referral_source
organic    114
other      103
ads         98
event       96
partner     89
Name: count, dtype: int64

Column: plan_tier
plan_tier
Pro           178
Basic         168
Enterprise    154
Name: count, dtype: int64


In [47]:
#account id column
cols = ['account_id', 'account_name'] #account_id has 500 unique values, same as from the subscription_table; every account_id has a unique account_name

for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(account[col].nunique(dropna=False))


Column: account_id
500

Column: account_name
500


In [48]:
account[['seats', 'account_id']].groupby('account_id').sum().sort_values('seats', ascending=False).head(10)

,seats
account_id,
A-56962b,163
A-18793f,148
A-d4e0d4,117
A-1f0636,105
A-5c046d,93
A-66224b,92
A-40906c,92
A-5a215a,87
A-4814a3,87


In [49]:
# seats will be more in subscription table because it contains past records of the same account, while account table only contains the latest record for each account
subscription[['seats', 'account_id']].groupby('account_id').sum().sort_values('seats', ascending=False).head(10)

,seats
account_id,
A-18793f,1628
A-5a215a,1566
A-d4e0d4,1170
A-1f0636,1155
A-56962b,1141
A-66224b,1123
A-40906c,1104
A-5b1bcd,1095
A-4814a3,957


#### C. Feature Usage

In [50]:
# feature usage categorical columns

cols = ['feature_name']

for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(feature_usage[col].value_counts(dropna=False))


Column: feature_name
feature_name
feature_12    659
feature_32    659
feature_6     655
feature_17    651
feature_34    650
feature_26    649
feature_36    648
feature_31    644
feature_11    643
feature_20    643
feature_24    643
feature_38    643
feature_2     642
feature_29    640
feature_15    640
feature_37    636
feature_22    636
feature_33    635
feature_10    634
feature_39    631
feature_1     629
feature_30    629
feature_4     625
feature_9     624
feature_28    623
feature_16    622
feature_40    621
feature_7     619
feature_25    615
feature_13    612
feature_27    611
feature_14    600
feature_3     598
feature_8     596
feature_21    591
feature_19    589
feature_18    588
feature_35    587
feature_5     577
feature_23    563
Name: count, dtype: int64


In [51]:
# feature usage id columns
cols = ['usage_id', 'subscription_id'] #usage_id is unique for every record (25.000) & subscription_id here is less than the subscription table because not every subscription has usage record, and some subscription has multiple usage records
for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(feature_usage[col].nunique(dropna=False))


Column: usage_id
25000

Column: subscription_id
4967


In [52]:
feature_usage.shape

(25000, 8)

#### D. Support Ticket

In [53]:
# support ticket categorical and scalar columns

cols = ['priority', 'satisfaction_score']

for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(support_ticket[col].value_counts(dropna=False))


Column: priority
priority
urgent    514
high      510
medium    491
low       485
Name: count, dtype: int64

Column: satisfaction_score
satisfaction_score
NaN    825
4.0    405
3.0    396
5.0    374
Name: count, dtype: int64


In [54]:
# support ticket unique columns

cols = ['ticket_id', 'account_id']
for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(support_ticket[col].nunique(dropna=False))


Column: ticket_id
2000

Column: account_id
492


In [55]:
support_ticket.shape

(2000, 9)

#### E. Churn Events

In [56]:
# churn events categorical columns

cols = ['reason_code']

for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(churn_event[col].value_counts(dropna=False))


Column: reason_code
reason_code
features      114
support       104
budget        104
unknown        95
competitor     92
pricing        91
Name: count, dtype: int64


In [57]:
# churn event id columns

cols = ['churn_event_id', 'account_id']

for col in cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(churn_event[col].nunique(dropna=False))


Column: churn_event_id
600

Column: account_id
352


In [58]:
# relationship between churn_event_id and account_id; one account can have multiple subscription and churn events
churn_event[['churn_event_id', 'account_id']].groupby('account_id').count().sort_values('churn_event_id', ascending=False).head(10)

,churn_event_id
account_id,
A-0a62f5,5
A-180abf,5
A-ee42db,4
A-0baac2,4
A-0cc442,4
A-ae052e,4
A-37f969,4
A-425e76,4
A-68f37c,4


In [59]:
churn_event[churn_event['account_id'] == 'A-0a62f5']

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
9,C-9328ba,A-0a62f5,2024-10-10,features,0.00,False,False,False,switched to competitor
151,C-b5bb09,A-0a62f5,2024-09-27,pricing,90.90,False,False,False,switched to competitor
177,C-7e1bbd,A-0a62f5,2024-12-18,budget,0.00,True,False,False,switched to competitor
242,C-b3d895,A-0a62f5,2024-12-27,support,43.60,False,False,False,too expensive
494,C-105c4c,A-0a62f5,2024-12-07,budget,21.95,False,True,False,NaN


In [60]:
subscription[subscription['account_id'] == 'A-0a62f5']

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
761,S-561850,A-0a62f5,2024-10-23,NaT,Pro,16,784,9408,False,False,False,False,annual,True
2836,S-d8c527,A-0a62f5,2024-12-21,NaT,Basic,25,475,5700,False,False,False,False,monthly,True
3636,S-e7a190,A-0a62f5,2024-12-06,NaT,Basic,22,418,5016,False,False,False,False,annual,True
4153,S-c62bb7,A-0a62f5,2024-09-16,NaT,Enterprise,16,3184,38208,False,False,False,False,monthly,True


In [61]:
churn_event.shape

(600, 9)

## Outlier Detection, Date Sanity and Flag Consistency

In [62]:
subscription[subscription['seats'] <= 0]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag


In [63]:
# already checked there's no negative mrr and arr amount
subscription[(subscription['mrr_amount'] <= 0) & (subscription['arr_amount'] <= 0) & (subscription['is_trial'] == 1)] 

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
2,S-51c0d1,A-659280,2024-11-25,NaT,Enterprise,62,0,0,True,True,False,False,annual,False
11,S-79a9e1,A-3a5ad8,2024-12-25,NaT,Enterprise,22,0,0,True,False,False,False,annual,True
14,S-428e9a,A-8ae3fc,2024-10-12,NaT,Enterprise,88,0,0,True,False,False,False,monthly,True
16,S-13939b,A-86902e,2023-07-21,2024-08-03,Pro,34,0,0,True,False,False,True,monthly,True
20,S-c27134,A-dec005,2024-03-31,NaT,Pro,21,0,0,True,False,False,False,monthly,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4959,S-409beb,A-7988d1,2024-03-12,NaT,Basic,33,0,0,True,False,False,False,annual,True
4966,S-e74ca1,A-59f724,2024-08-24,NaT,Basic,3,0,0,True,False,False,False,monthly,False
4970,S-155934,A-4a4c2d,2024-04-16,NaT,Pro,54,0,0,True,False,False,False,annual,True
4971,S-535bd1,A-d792a6,2023-03-18,NaT,Pro,9,0,0,True,False,False,False,monthly,False


In [64]:
# all rows with 0 mrr & arr amount are trial
subscription[subscription['is_trial'] == 1]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
2,S-51c0d1,A-659280,2024-11-25,NaT,Enterprise,62,0,0,True,True,False,False,annual,False
11,S-79a9e1,A-3a5ad8,2024-12-25,NaT,Enterprise,22,0,0,True,False,False,False,annual,True
14,S-428e9a,A-8ae3fc,2024-10-12,NaT,Enterprise,88,0,0,True,False,False,False,monthly,True
16,S-13939b,A-86902e,2023-07-21,2024-08-03,Pro,34,0,0,True,False,False,True,monthly,True
20,S-c27134,A-dec005,2024-03-31,NaT,Pro,21,0,0,True,False,False,False,monthly,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4959,S-409beb,A-7988d1,2024-03-12,NaT,Basic,33,0,0,True,False,False,False,annual,True
4966,S-e74ca1,A-59f724,2024-08-24,NaT,Basic,3,0,0,True,False,False,False,monthly,False
4970,S-155934,A-4a4c2d,2024-04-16,NaT,Pro,54,0,0,True,False,False,False,annual,True
4971,S-535bd1,A-d792a6,2023-03-18,NaT,Pro,9,0,0,True,False,False,False,monthly,False


In [65]:
#these are outliers, a subscription record cannot be both upgrade and downgrade
subscription[(subscription['mrr_amount'] == 0) & (subscription['arr_amount'] == 0) & (subscription['upgrade_flag'] == 1) & (subscription['downgrade_flag'] == 1)]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
32,S-2877e6,A-95b24a,2024-06-28,NaT,Enterprise,38,0,0,True,True,True,False,monthly,True
1711,S-149409,A-0cc442,2024-09-15,2024-10-28,Pro,45,0,0,True,True,True,True,monthly,True
4918,S-ff2510,A-80eeb6,2024-09-14,NaT,Pro,46,0,0,True,True,True,False,annual,True


In [66]:
subscription[subscription['account_id'] == 'A-95b24a'].sort_values('start_date')

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
3516,S-0ed7be,A-95b24a,2023-07-14,NaT,Pro,47,2303,27636,False,False,False,False,monthly,False
1865,S-209456,A-95b24a,2023-07-18,NaT,Pro,38,1862,22344,False,False,False,False,annual,True
4835,S-1f505e,A-95b24a,2023-07-30,NaT,Basic,38,722,8664,False,True,False,False,annual,True
2214,S-444c6a,A-95b24a,2023-10-07,NaT,Basic,38,722,8664,False,False,False,False,annual,False
2793,S-a29ba3,A-95b24a,2023-10-12,NaT,Enterprise,38,7562,90744,False,False,False,False,monthly,False
1549,S-0865f1,A-95b24a,2023-12-10,NaT,Basic,38,722,8664,False,False,False,False,annual,True
58,S-b9ff15,A-95b24a,2024-01-08,NaT,Enterprise,43,8557,102684,False,False,False,False,annual,False
665,S-799da2,A-95b24a,2024-06-27,NaT,Pro,38,1862,22344,False,False,False,False,monthly,True
32,S-2877e6,A-95b24a,2024-06-28,NaT,Enterprise,38,0,0,True,True,True,False,monthly,True
4649,S-891976,A-95b24a,2024-09-12,NaT,Basic,38,722,8664,False,True,False,False,monthly,True


In [67]:
subscription[(subscription['mrr_amount'] == 0) & (subscription['arr_amount'] == 0) & (subscription['upgrade_flag'] == 0) & (subscription['downgrade_flag'] == 0) &(subscription['is_trial'] == 0)]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag


In [68]:
subscription[(subscription['mrr_amount'] == 0) & (subscription['arr_amount'] == 0) & (subscription['upgrade_flag'] == 1) & (subscription['downgrade_flag'] == 0) & (subscription['is_trial'] == 0)]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag


In [69]:
#these are accounts who downgraded into free trials
subscription[(subscription['mrr_amount'] == 0) & (subscription['arr_amount'] == 0) & (subscription['upgrade_flag'] == 0) & (subscription['downgrade_flag'] == 1)]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
190,S-13ca37,A-6843f2,2024-12-21,NaT,Pro,21,0,0,True,False,True,False,monthly,True
233,S-3f82bc,A-d5a319,2023-10-30,NaT,Basic,15,0,0,True,False,True,False,monthly,True
318,S-17cfc4,A-592832,2024-12-01,NaT,Pro,18,0,0,True,False,True,False,monthly,True
689,S-7d0dea,A-69b156,2024-11-16,NaT,Pro,7,0,0,True,False,True,False,monthly,True
1448,S-be079b,A-6da850,2024-04-29,NaT,Basic,5,0,0,True,False,True,False,monthly,False
1458,S-814624,A-671f31,2024-12-28,NaT,Enterprise,82,0,0,True,False,True,False,annual,True
1695,S-e53224,A-d82ef9,2024-08-25,NaT,Pro,26,0,0,True,False,True,False,annual,True
2014,S-28e491,A-337bc1,2024-09-28,NaT,Pro,30,0,0,True,False,True,False,annual,True
2262,S-060019,A-d535f6,2024-10-18,NaT,Basic,19,0,0,True,False,True,False,annual,True
2272,S-bc7cab,A-cc8c8f,2024-12-19,NaT,Basic,48,0,0,True,False,True,False,annual,True


In [70]:
subscription[(subscription['mrr_amount']) > (subscription['arr_amount'])]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag


In [71]:
subscription[(subscription['mrr_amount']*12) != (subscription['arr_amount'])]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag


In [72]:
account[account['seats'] <= 0]

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag


In [73]:
# the second row is likely a failed attempt because error_count > 0; but the 1st row is likely a logging bug because usage_duration_secs is 0 (possibly because it's a beta feature)
feature_usage[feature_usage['usage_count'] <= 0]

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
2875,2876,S-9925b9,2023-08-01,feature_25,0,0,0,False
3327,3328,S-c98905,2023-03-15,feature_34,0,0,2,True


In [74]:
feature_usage[feature_usage['usage_duration_secs'] <= 0]

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
2875,2876,S-9925b9,2023-08-01,feature_25,0,0,0,False
3327,3328,S-c98905,2023-03-15,feature_34,0,0,2,True


In [75]:
support_ticket[support_ticket['submitted_at'] > support_ticket['closed_at']]

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag


In [76]:
support_ticket[support_ticket['resolution_time_hours'] <= 0]

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag


In [77]:
support_ticket[support_ticket['first_response_time_minutes'] <= 0]

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag


In [78]:
# churn flag is count on a subscription basis, while churn event is counted on an account basis
subscription['churn_flag'].value_counts()

churn_flag
False    4510
True      486
Name: count, dtype: int64

In [79]:
churn_event['is_reactivation'].value_counts() #61 customers actually came back, while 539 left for good

is_reactivation
False    539
True      61
Name: count, dtype: int64

In [80]:
# count churn events per account (excluding reactivations)
churn_counts = churn_event[churn_event['is_reactivation'] == False].groupby('account_id')['churn_event_id'].count().reset_index()
churn_counts.columns = ['account_id', 'churn_event_count']

# join with subscription churn flag
subscription.groupby('account_id')['churn_flag'].sum().reset_index().merge(
    churn_counts,
    on='account_id',
    how='outer'
).query('churn_flag != churn_event_count').sort_values('account_id')

,account_id,churn_flag,churn_event_count
0,A-00bed1,0,1.0
1,A-00cac8,0,NaN
2,A-0158bb,0,NaN
4,A-019782,1,NaN
7,A-02fac6,0,1.0
...,...,...,...
494,A-fdfc91,1,NaN
495,A-fe79a5,2,NaN
496,A-ff3c73,1,2.0
497,A-ff79f2,2,NaN


In [81]:
# get churn flag sum per account from subscription
sub_churn = subscription.groupby('account_id').agg(
    total_subscriptions=('subscription_id', 'count'),
    churned_subscriptions=('churn_flag', 'sum')
).reset_index()

# get churn event count per account
event_churn = churn_event[churn_event['is_reactivation'] == False].groupby('account_id')['churn_event_id'].count().reset_index()
event_churn.columns = ['account_id', 'churn_event_count']

# join both
sub_churn.merge(
    event_churn,
    on='account_id',
    how='outer'
).sort_values('churned_subscriptions', ascending=False)
# data pipeline issue: some accounts have churn events but no churn flag in subscription table, likely because the subscription table only captures the latest subscription record for each account, while the churn event table captures all churn events on an account basis
# the churn event table will just be used to find out more about churn reasons while the churn fact table will be based on the subscription table

,account_id,total_subscriptions,churned_subscriptions,churn_event_count
181,A-5a215a,18,6,NaN
46,A-157070,16,4,2.0
464,A-eb7c38,15,4,1.0
42,A-13466a,6,4,1.0
428,A-dbc825,16,4,2.0
...,...,...,...,...
472,A-f19b24,3,0,NaN
473,A-f1f639,4,0,1.0
474,A-f25509,5,0,2.0
1,A-00cac8,9,0,NaN


### Action Needed

In [82]:
# dropping the outliers from subscription table
subscription = subscription[~((subscription['mrr_amount'] == 0) & (subscription['arr_amount'] == 0) & (subscription['upgrade_flag'] == 1) & (subscription['downgrade_flag'] == 1))]

In [83]:
subscription[(subscription['mrr_amount'] == 0) & (subscription['arr_amount'] == 0) & (subscription['upgrade_flag'] == 1) & (subscription['downgrade_flag'] == 1)]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag


In [84]:
# dropping the outliers from feature usage table
feature_usage = feature_usage[~((feature_usage['usage_count'] <= 0) | (feature_usage['error_count'] <= 0))]

In [85]:
feature_usage[feature_usage['usage_count'] <= 0]

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature


## Final: Export Data to PgAdmin4

In [86]:
# subscription.to_sql('subscription', con=engine, if_exists='replace', index=False)

In [91]:
account.to_sql('account', con=engine, if_exists='replace', index=False)

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)

In [88]:
# feature_usage.to_sql('feature_usage', con=engine, if_exists='replace', index=False)

In [89]:
# support_ticket.to_sql('support_ticket', con=engine, if_exists='replace', index=False)

In [90]:
# churn_event.to_sql('churn_event', con=engine, if_exists='replace', index=False)